### 1. Separar

In [ ]:
# arvore com listas: ["var", "p"], ["-", no], ["and"/"or", esq, dir]
def separar(texto):
    tokens = []
    pos = 0
    while pos < len(texto):
        c = texto[pos]
        if c.isspace():
            pos += 1
        elif c in "-()":
            tokens.append(c)
            pos += 1
        elif c.isascii() and (c.isalpha() or c == "_"):
            inicio = pos
            pos += 1
            while pos < len(texto) and texto[pos].isascii() and (texto[pos].isalnum() or texto[pos] == "_"):
                pos += 1
            tokens.append(texto[inicio:pos])
        else:
            raise ValueError(f"Caractere inválido '{c}' na posição {pos}.")
    return tokens

### 2. Montar a árvore

In [ ]:
# pos fica numa lista para as funcoes conseguirem mudar o valor
def montar_arvore(tokens):
    if len(tokens) == 0:
        raise ValueError("A expressão está vazia.")
    pos = [0]
    arvore = ler_or(tokens, pos)
    if pos[0] != len(tokens):
        raise ValueError(f"Token inesperado '{tokens[pos[0]]}' na posição de token {pos[0]}; esperava um operador ou o fim.")
    return arvore


def ler_or(tokens, pos):
    no = ler_and(tokens, pos)
    while pos[0] < len(tokens) and tokens[pos[0]] == "or":
        pos[0] += 1
        no = ["or", no, ler_and(tokens, pos)]
    return no


def ler_and(tokens, pos):
    no = ler_nao(tokens, pos)
    while pos[0] < len(tokens) and tokens[pos[0]] == "and":
        pos[0] += 1
        no = ["and", no, ler_nao(tokens, pos)]
    return no


def ler_nao(tokens, pos):
    if pos[0] < len(tokens) and tokens[pos[0]] == "-":
        pos[0] += 1
        return ["-", ler_nao(tokens, pos)]
    return ler_primario(tokens, pos)


def ler_primario(tokens, pos):
    if pos[0] == len(tokens):
        raise ValueError("Operando ausente no fim da expressão.")
    t = tokens[pos[0]]
    if t == "(":
        pos[0] += 1
        no = ler_or(tokens, pos)
        if pos[0] == len(tokens) or tokens[pos[0]] != ")":
            raise ValueError("Parêntese de fechamento ')' ausente.")
        pos[0] += 1
        return no  # parentese vira galho
    if t == "and" or t == "or" or t == ")":
        raise ValueError(f"Esperava uma variável ou '('; encontrei '{t}' na posição de token {pos[0]}.")
    pos[0] += 1
    return ["var", t]

### 3. Simplificar

In [ ]:
def simplificar(no, negado=False):
    if no[0] == "var":
        if negado:
            return ["-", no]
        return no
    if no[0] == "-":
        # dois - seguidos se cancelam (dupla negacao)
        return simplificar(no[1], not negado)
    op = no[0]
    if negado:
        # De Morgan: troca and por or e leva o nao para os dois lados
        if op == "and":
            op = "or"
        else:
            op = "and"
    return [op, simplificar(no[1], negado), simplificar(no[2], negado)]

### 4. Reescrever

In [ ]:
def escrever(no, prioridade_pai=0):
    if no[0] == "var":
        prioridade = 4
    elif no[0] == "-":
        prioridade = 3
    elif no[0] == "and":
        prioridade = 2
    else:
        prioridade = 1

    if no[0] == "var":
        texto = no[1]
    elif no[0] == "-":
        texto = "-" + escrever(no[1], prioridade)
    else:
        texto = escrever(no[1], prioridade) + " " + no[0] + " " + escrever(no[2], prioridade)

    if prioridade < prioridade_pai:  # so poe () se precisar
        texto = "(" + texto + ")"
    return texto

### 5. Juntar tudo

In [ ]:
def simplify(texto):
    tokens = separar(texto)
    arvore = montar_arvore(tokens)
    return escrever(simplificar(arvore))

### Rodar

In [ ]:
# exemplos: -(p and (q or -r))   ou   p q  (esse da erro)
expressao = input("Digite a expressão: ")
try:
    print("Simplificada:", simplify(expressao))
except ValueError as erro:
    print("Erro:", erro)